Vector Search Fundamentals

In [ ]:
# installing all the necessary libraries required
!pip install faiss-cpu sentence-transformers umap-learn --quiet

import numpy as np , faiss, time
import matplotlib.pyplot as plt , matplotlib.patches as mpatches

from sentence_transformers import SentenceTransformer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 59.1 MB/s eta 0:00:00


TO prove that embedding encode meanings geometrically

Goal - To check whether similar sentence cluster together in vector space .

In [ ]:
model=SentenceTransformer('all-MiniLM-L6-v2')  #384 dimension embedding

sentence =[
    #group -1 Royalty
    'The king rule the kindom wisely',
    'The queen addressed her royal court',
    #group-2 Animal
    'The dog played fetch in the park',
    'The puppy chased a ball outside',
    #group -3 programming
    'Python is used for Machine Learning',
    "javascript runs in the browser",
    #wildcard
    'the stock market crashed yesterday'
]

embedding=model.encode(sentence)
print(f'Shape of embeddings:{embedding.shape}')
print(f'Each sentence is now a vector {embedding.shape[1] }numbers')
print(f'First 5 vales od sentence {embedding[0][:5].round(4)}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of embeddings:(7, 384)
Each sentence is now a vector 384numbers
First 5 vales od sentence [-0.0318  0.0848 -0.0133  0.0112 -0.0461]


In [ ]:
# computing pairwaise cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix=cosine_similarity(embedding)

print('cosine Similarity Matrix:')
# print(sim_matrix)
print('(1.0 = identical meaning , 0.0 - unrealted , -1.0 - opposite')
print()

for i , s1 in enumerate(sentence):
  for j , s2 in enumerate(sentence):

    if i < j :
      sim=sim_matrix[i][j]
      # print(sim)
      bar=' ' * int(sim * 20)
      print(f'{sim:.3f} {bar:<20} | {s1[:30]} <-> {s2[:30]}')

cosine Similarity Matrix:
(1.0 = identical meaning , 0.0 - unrealted , -1.0 - opposite

0.228                      | The king rule the kindom wisel <-> The queen addressed her royal 
0.008                      | The king rule the kindom wisel <-> The dog played fetch in the pa
-0.021                      | The king rule the kindom wisel <-> The puppy chased a ball outsid
-0.025                      | The king rule the kindom wisel <-> Python is used for Machine Lea
-0.000                      | The king rule the kindom wisel <-> javascript runs in the browser
-0.006                      | The king rule the kindom wisel <-> the stock market crashed yeste
0.004                      | The queen addressed her royal  <-> The dog played fetch in the pa
0.042                      | The queen addressed her royal  <-> The puppy chased a ball outsid
0.002                      | The queen addressed her royal  <-> Python is used for Machine Lea
0.060                      | The queen addressed her 

In [ ]:
print(embedding[0][:5])

[-0.03179544  0.08484644 -0.01326337  0.01116466 -0.04610432]


Excercise-2 The Curse of Dimensionality - made real

Goal - mathematically that in high dimension , all vectors become equidistant

predict first- what will happens to t he std/mean ratio as dimension increase?

In [ ]:
def measure_distance_concentration(n_vectors=1000, dims_list=[2, 10, 50, 128, 384, 768, 1536]):
    results = []
    for d in dims_list:
        # Generate random unit vectors (normalized)
        vecs = np.random.randn(n_vectors, d).astype(np.float32)
        vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)

        # Compute all pairwise L2 distances
        query=vecs[0:1]
        print(query.shape)
        print(query)
        rest=vecs[1:]
        # print(rest)
        dists=np.linalg.norm(rest-query,axis=1)


print(measure_distance_concentration())

(1, 2)
[[-0.987237   -0.15925848]]
(1, 10)
[[-0.4798881  -0.06878017 -0.24015494 -0.4275108  -0.23665994 -0.19293548
   0.46385193  0.10358399 -0.06845465 -0.44802952]]
(1, 50)
[[-0.06962459 -0.15336064 -0.1523567   0.2280017  -0.06120497  0.13485436
   0.09189135 -0.05156939 -0.16425051  0.08788825 -0.14228186  0.16322808
   0.14864041 -0.20247373 -0.00252486 -0.02430555  0.11986244 -0.03430648
  -0.21221638 -0.00089628 -0.27309632  0.14448291  0.05274863 -0.17324707
  -0.06392173  0.09813224 -0.02786388 -0.15662204 -0.00256817  0.13490567
  -0.13216461 -0.3381395  -0.03692162  0.07348868  0.12855074  0.21663074
   0.04158946 -0.10226403  0.07270454  0.17715085 -0.06154262  0.23480597
  -0.10320317  0.2697619   0.00700324 -0.16560364 -0.02967562  0.0076209
   0.19679467 -0.10814959]]
(1, 128)
[[-0.17368065  0.01688701 -0.15160699  0.05378727  0.03149463 -0.09646149
  -0.05536555  0.02475789  0.06345548 -0.1212809   0.09028643  0.03453164
  -0.03106048  0.04813632  0.10418922 -0.050051

In [ ]:
def measure_distance_concentration(n_vectors=1000, dims_list=[2, 10, 50, 128, 384, 768, 1536]):
    results = []
    for d in dims_list:
        # Generate random unit vectors (normalized)
        vecs = np.random.randn(n_vectors, d).astype(np.float32)
        vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)

        # Compute all pairwise L2 distances
        query=vecs[0:1]
        rest=vecs[1:]
        dists=np.linalg.norm(rest-query,axis=1)

        mean_dist=dists.mean()
        std_dist=dists.std()
        ratio=std_dist / mean_dist #how muxh vriation

        results.append({
            'dims': d,
            'mean': round(mean_dist, 4),
            'std': round(std_dist, 4),
            'variation_ratio': round(ratio, 4),
            'min': round(dists.min(), 4),
            'max': round(dists.max(), 4)
        })
    return results
results = measure_distance_concentration()

print(f'{'Dims':<8} {'Mean dist':<12} {'Std dev':<10} {'Std/Mean':<12} {'Min':<8} {'Max':<8}')
print('-' * 60)
for r in results:
    print(f"{r['dims']:<8} {r['mean']:<12} {r['std']:<10} {r['variation_ratio']:<12} {r['min']:<8} {r['max']:<8}")

print()
print('KEY INSIGHT: As dims increase, Std/Mean → 0.')
print('This means ALL points look equally close to the query.')
print('This is WHY brute force search in 1536D is so hard — signal is tiny.')

measure_distance_concentration()

Dims     Mean dist    Std dev    Std/Mean     Min      Max     
------------------------------------------------------------
2        1.277899980545044 0.61080002784729 0.4779999852180481 0.0035000001080334187 2.0     
10       1.3968000411987305 0.23340000212192535 0.1670999974012375 0.6446999907493591 1.9414000511169434
50       1.4038000106811523 0.10040000081062317 0.07150000333786011 1.0835000276565552 1.6601999998092651
128      1.4143999814987183 0.06279999762773514 0.04439999908208847 1.2121000289916992 1.636199951171875
384      1.4140000343322754 0.034699998795986176 0.02449999935925007 1.2842999696731567 1.5335999727249146
768      1.4128999710083008 0.026100000366568565 0.01850000023841858 1.333799958229065 1.4958000183105469
1536     1.4127999544143677 0.01850000023841858 0.013100000098347664 1.3609000444412231 1.478600025177002

KEY INSIGHT: As dims increase, Std/Mean → 0.
This means ALL points look equally close to the query.
This is WHY brute force search in 1536D is so

[{'dims': 2,
  'mean': np.float32(1.2791),
  'std': np.float32(0.624),
  'variation_ratio': np.float32(0.4879),
  'min': np.float32(0.0033),
  'max': np.float32(2.0)},
 {'dims': 10,
  'mean': np.float32(1.3939),
  'std': np.float32(0.2379),
  'variation_ratio': np.float32(0.1707),
  'min': np.float32(0.6178),
  'max': np.float32(1.9205)},
 {'dims': 50,
  'mean': np.float32(1.4083),
  'std': np.float32(0.0984),
  'variation_ratio': np.float32(0.0699),
  'min': np.float32(1.1076),
  'max': np.float32(1.6906)},
 {'dims': 128,
  'mean': np.float32(1.4135),
  'std': np.float32(0.0643),
  'variation_ratio': np.float32(0.0455),
  'min': np.float32(1.1819),
  'max': np.float32(1.5794)},
 {'dims': 384,
  'mean': np.float32(1.4133),
  'std': np.float32(0.0372),
  'variation_ratio': np.float32(0.0263),
  'min': np.float32(1.2931),
  'max': np.float32(1.5085)},
 {'dims': 768,
  'mean': np.float32(1.4129),
  'std': np.float32(0.0251),
  'variation_ratio': np.float32(0.0177),
  'min': np.float32(1.3

# **Excercise-3 The 3 dsitance metrics where each one FAILS**

Goal - Construct a case where cosine succeeds but L2 fails , and vice vera

In [ ]:
#create 3 vector that reveal the difference between metrics

import numpy as np

v1=np.array([1.0,0.0])  #pointing right , lenght 1
v2=np.array([10.0,0.0])  #pointing right , length 10
v3=np.array([0.0,1.0])      #pointing up , lengt 1

def cosine_sim(a,b):
  return np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b))

def l2_dist(a,b):
  return np.linalg.norm(a-b)

def dot_product(a,b):
  return np.dot(a,b)

print("Scenaria: Querry=v1 [1,0]. which is more similar: v2[10,0] or v3[0,1]?")
print()
print('v1 and v2 point in the same direction(seam meaning) but different magnitude')
print('v1 and v3 point in DIFFERENT direction (different meaning)\n')

print(f"Cosine(v1,v2)={cosine_sim(v1,v2):.3f} <-- v2 ia SAME direction")
print(f"Cosine(v1,v3)={cosine_sim(v1,v3):.3f} <-- v3 is DIFFERENT direction")
print(f'Consine correctly identifies v2 as more similar \n')

print(f"L2(v1,v2)={l2_dist(v1,v2):.3f}  <-- v2 is far away (magnitude diff)")
print(f"L2(v1,v3)={l2_dist(v1,v3):.3f} <-- v3 is  closer in raw space")
print(f'L2 incorrectly ranks v3 as more similar! \n')

print(f'Dot(v1,v2)={dot_product(v1,v2):.3f}  <- dominatede by magnitude')
print(f"Dot(v1,v3)={dot_product(v1,v3):.3f}  <-- correctly near zero")
print(f'Dot correctly finds v2 but for wrong reason (magnitude and direction)\n')

print("="*60)
print("LESSON:FAISS IndexFlatL2 on un-normalised embedding WILL")
print("rank long documents over short ones regaradless of meaning")
print('Always normalise if using L2 for semantic search')
print('Alternatively, use IndexFlatIP (Inner Product) with normalized vectors, which is equivalent to cosine similarity.')



Scenaria: Querry=v1 [1,0]. which is more similar: v2[10,0] or v3[0,1]?

v1 and v2 point in the same direction(seam meaning) but different magnitude
v1 and v3 point in DIFFERENT direction (different meaning)

Cosine(v1,v2)=1.000 <-- v2 ia SAME direction
Cosine(v1,v3)=0.000 <-- v3 is DIFFERENT direction
Consine correctly identifies v2 as more similar 

L2(v1,v2)=9.000  <-- v2 is far away (magnitude diff)
L2(v1,v3)=1.414 <-- v3 is  closer in raw space
L2 incorrectly ranks v3 as more similar! 

Dot(v1,v2)=10.000  <- dominatede by magnitude
Dot(v1,v3)=0.000  <-- correctly near zero
Dot correctly finds v2 but for wrong reason (magnitude and direction)

LESSON:FAISS IndexFlatL2 on un-normalised embedding WILL
rank long documents over short ones regaradless of meaning
Always normalise if using L2 for semantic search
Alternatively, use IndexFlatIP (Inner Product) with normalized vectors, which is equivalent to cosine similarity.


# **Real World Test : Does this actually happens with sentence embeddings?**

In [ ]:
!pip install Sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

short_exact="python  programming Language"

long_noisy='''Python is a high-level , interpreted , general-purpose
programming language known for its readable syntax.
It was create by guido van Rossum and first released in 1991.
Python emhasizes code readablility with the use of significant
indentation.'''

unrelated='The French Revoltion began in 1789 with the storming of the bastille.'


query='What is python?'
model=SentenceTransformer('all-MiniLM-L6-v2')  #384 dimension embeddings

vecs=model.encode([query,short_exact,long_noisy,unrelated])
q,s1,s2,s3=vecs[0],vecs[1],vecs[2],vecs[3]

print(vecs.shape)
print(f'Query: {query} \n')
print("---WITHOUT NORMALISAION (raw L2) ---")
print(f'L2 to short exact match: {l2_dist(q,s1)}')
print(f'L2 to lomg noisy match: {l2_dist(q,s2)}')
print(f'L2 to unrelated match: {l2_dist(q,s3)}')

#normalse
qn=q / np.linalg.norm(q)
s1n=s1 / np.linalg.norm(s1)
s2n=s2 / np.linalg.norm(s2)
s3n=s3 / np.linalg.norm(s3)

print()
print('---WITH normalization (cosine equivalent) ---')
print(f'Cosine to short exact match :{cosine_sim(q,s1)}')
print(f'Cosine to Long noisy match :{cosine_sim(q,s2)}')
print(f'Cosine to unrelated match :{cosine_sim(q,s3)}')

# print() not L2_dist instead dot_product after normalisation cosine ==dot
# print("---WITHOUT NORMALISAION (raw L2) ---")
# print(f'L2 to short exact match: {l2_dist(qn,s1n)}')
# print(f'L2 to lomg noisy match: {l2_dist(qn,s2n)}')
# print(f'L2 to unrelated match: {l2_dist(qn,s3n)}')

print()
print('OBSERVE:Doess L2 unfairly rank the long noisy match higher')
print('this is the magnitude bais bug in real Rag System.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(4, 384)
Query: What is python? 

---WITHOUT NORMALISAION (raw L2) ---
L2 to short exact match: 0.6105924844741821
L2 to lomg noisy match: 0.6466696262359619
L2 to unrelated match: 1.4462506771087646

---WITH normalization (cosine equivalent) ---
Cosine to short exact match :0.8135884404182434
Cosine to Long noisy match :0.7909092307090759
Cosine to unrelated match :-0.045820459723472595

OBSERVE:Doess L2 unfairly rank the long noisy match higher
this is the magnitude bais bug in real Rag System.


# **OBSERVATION:**

MiniLM normalizes embeddings by default.
The magnitude bias bug exists but this model hides it.
To reproduce it clearly: use raw embeddings or a model
without output normalization.
ALWAYS check whether your embedding model normalizes
outputs — don't assume.

# **Excerice-4 Prove why brute force doesnt scale**

Goal : Measure real query latency vs dataset size , Find the breaking point

In [ ]:
d = 384  # Embedding dimension (e.g., from MiniLM model)
results = []  # Store (n, latency) pairs

for i in [1_000, 10_000, 50_000, 100_000, 500_000]:
    # Generate i random vectors of dimension d
    vecs = np.random.randn(i, d).astype(np.float32)
    faiss.normalize_L2(vecs)  # Normalize to unit length (for cosine similarity)

    # Generate a random query vector
    query = np.random.randn(1, d).astype(np.float32)
    faiss.normalize_L2(query)

    # Create brute-force L2 index
    index = faiss.IndexFlatL2(d)
    index.add(vecs)  # Add vectors to index

    # Warm up (avoid caching effects)
    index.search(query, 5)

    # Time 10 searches
    start = time.time()
    for _ in range(10):
        index.search(query, 5)
    elapsed_ms = (time.time() - start) / 10 * 1000  # Average ms per query

    results.append((i, elapsed_ms))
    print(f'n={i:>7,} | brute force latency: {elapsed_ms:.2f}ms')

print()
print('if brute force at n=500k is already slow on colab')
print('imagine n=10m in production. ANN is not optimal - it is required.')

n=  1,000 | brute force latency:0.07ms

if brute force at n=500k is already slow on colab
imagine n=10m in prduction.ANN is not optimal- it is required.
n= 10,000 | brute force latency:1.48ms

if brute force at n=500k is already slow on colab
imagine n=10m in prduction.ANN is not optimal- it is required.
n= 50,000 | brute force latency:7.60ms

if brute force at n=500k is already slow on colab
imagine n=10m in prduction.ANN is not optimal- it is required.
n= 10,000 | brute force latency:1.27ms

if brute force at n=500k is already slow on colab
imagine n=10m in prduction.ANN is not optimal- it is required.
n=500,000 | brute force latency:118.01ms

if brute force at n=500k is already slow on colab
imagine n=10m in prduction.ANN is not optimal- it is required.


What it does:
Measures query latency (in milliseconds) for searching in datasets of 1K, 10K, 50K, 100K, and 500K vectors.

Uses normalized random vectors (384-dimensional, like MiniLM) to simulate embeddings.

Demonstrates that brute-force search becomes slow at scale (e.g., 500K vectors).

Concludes that Approximate Nearest Neighbor (ANN) methods are required for large-scale production systems.

**Key takeaway:**

Brute-force (IndexFlatL2) is exact but slow at scale.  For large datasets (e.g., 10M+), ANN indexes (like IVF, HNSW) are essential for low-latency search.